# Collaborative Filtering Recommender System - Project Part II 🤖💡

## Learning Objectives 🎯
By completing this assignment, you will:
1. **Implement a collaborative filtering recommender** using matrix factorization 🧮
2. **Learn gradient descent optimization** for machine learning models 📉
3. **Compare recommendation algorithms** against baseline approaches 📊
4. **Analyze business performance metrics** like revenue and conversion rates 💰
5. **Understand realistic training constraints** in recommendation systems 🔍

## Problem Overview 🌍
You will build and evaluate a Collaborative Filtering (CF) recommender system that learns customer preferences and item characteristics through low-dimensional embeddings. Your system will be trained on realistic customer interaction data and compared against simple baseline strategies.

**Key Challenge**: Unlike academic datasets, real recommender systems face the "cold start" problem where customers only interact with a small subset of available items during training, but the system must make predictions for all items. ❄️

## Evaluation Approach 📈
Your CF model will be evaluated on:
1. **κ(c,i) prediction accuracy**: How well it predicts purchase probabilities 🎯
2. **Revenue optimization**: Ability to recommend high-value items customers will buy 💵
3. **Baseline comparison**: Performance vs Random and Most Expensive strategies ⚖️
4. **Business metrics**: Total revenue, conversion rates, and customer value 📊

# Step 0: Initialize System

In [ ]:
#@title ⚙️ Setup: download project data and install packages
import os, subprocess, sys, shutil

repo_url = "https://github.com/eth-ainit-fs26/project.git"
branch = "week2"
project_dir = "/content/project/Project1"

# Always start fresh to avoid stale state
if os.path.exists("/content/project"):
    shutil.rmtree("/content/project")

subprocess.run(["git", "clone", "--branch", branch, "--single-branch", repo_url, "/content/project"], check=True)
os.chdir(project_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-p2.txt", "-q"], check=True)
print("✅ Setup complete — working directory:", os.getcwd())

In [ ]:
#@title 🔧 Environment Setup — run, don't modify
# ▶️ Run this to configure the environment and set up paths
# Configure environment and basic imports
from utils.recommender_systems_utils import setup_environment
setup_environment()

# Import scientific libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error
import itertools

In [ ]:
#@title 📦 Module Imports — run, don't modify
# ▶️ Run this to import all required modules
# Import project modules and utilities
from models.customer_registry import CustomerRegistry
from models.item_catalogue import ItemCatalogue
from models.transaction_registry import TransactionRegistry
from agents.project2_recommendation.recommender_agent import RecommenderAgent
from agents.project2_recommendation.random_recommender import RandomRecommenderAgent
from agents.project2_recommendation.most_expensive_recommender import MostExpensiveRecommenderAgent
from typing import List

# Import all CF analysis utilities
from utils.recommender_systems_utils import (
    plot_learning_curves,
    plot_kappa_prediction_quality,
    plot_training_data_coverage,
    plot_comprehensive_performance_analysis,
    print_comprehensive_analysis,
    detailed_agent_simulation,
    analyze_and_plot_kappa_prediction_quality,
    validate_todo_implementations,
    prepare_ground_truth_data,
    initialize_system_components
)


In [ ]:
#@title ⚙️ System Initialization — run, don't modify
# ▶️ Run this to initialize the database and load customer/item data
# Initialize system components with hyperparameters
customer_registry, item_catalogue, transaction_registry, hyperparams = initialize_system_components(
    n_iterations=300,
    n_simulation_days=100,
    k=5,
    items_per_customer=15
)

# Extract hyperparameters for convenience
N_ITERATIONS = hyperparams['N_ITERATIONS']
N_SIMULATION_DAYS = hyperparams['N_SIMULATION_DAYS']
K = hyperparams['K']
ITEMS_PER_CUSTOMER = hyperparams['ITEMS_PER_CUSTOMER']

## RecommenderAgent Implementation 🤖

### Overview 📋
You will implement a **Collaborative Filtering (CF) recommender** using logistic matrix factorization. The core idea is to learn low-dimensional embeddings that capture customer preferences and item characteristics, then use these to predict purchase probabilities and optimize recommendations for revenue. 💡

### Mathematical Foundation 📐

#### Core Concepts 🧮
- **Customer embedding**: $U_c \in \mathbb{R}^d$ - A d-dimensional vector representing customer c's preferences
- **Item embedding**: $V_i \in \mathbb{R}^d$ - A d-dimensional vector representing item i's characteristics  
- **Biases**: $b_c, b_i \in \mathbb{R}$ - Individual customer and item bias terms
- **Purchase probability**: $\kappa(c,i) = \sigma(U_c^T V_i + b_c + b_i)$ where $\sigma$ is the sigmoid function

#### Revenue Optimization Strategy 💰
The recommender selects items that maximize expected revenue:
$$\text{Expected Revenue}(c,i) = (\text{price}_i - \text{cost}_i) \times \kappa(c,i)$$

#### Learning Algorithm 📚
The model learns through gradient descent on logistic loss using customer feedback:
- **Positive feedback**: Customer purchased item → target = 1.0 ✅
- **Negative feedback**: Customer saw but didn't purchase → target = 0.0 ❌

### Key Methods to Implement 🔧
1. **`_kappa_hat()`**: Predict purchase probability using embeddings 🎯
2. **`recommend()`**: Select top-K items by expected revenue 🏆
3. **`_update_embeddings()`**: Update embeddings via gradient descent 🔄
4. **`update_from_session()`**: Process customer purchase feedback 📝

### TODO 1: Implement Purchase Probability Prediction

**Function**: `_kappa_hat(self, customer_id: int, item_id: int) -> float`

**Goal**: Predict the probability that customer c will purchase item i using learned embeddings.

**Mathematical Formula**: 
$$\hat{\kappa}(c,i) = \sigma(U_c^T V_i + b_c + b_i)$$

where $\sigma(x) = \frac{1}{1 + e^{-x}}$ is the sigmoid function.

**Implementation Instructions**:
1. Initialize customer/item embeddings if they don't exist (use `_init_customer` and `_init_item`)
2. Compute the dot product: $U_c^T V_i$
3. Add customer bias $b_c$ and item bias $b_i$  
4. Apply sigmoid function to get probability in [0,1]
5. Return the predicted purchase probability

**Hint**: Use the provided `_sigmoid()` helper method.

In [ ]:
def _kappa_hat(self, customer_id: int, item_id: int) -> float:
    """
    Predict the probability that customer will purchase item using learned embeddings.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Returns:
        float: Purchase probability in range [0, 1]
    """
    if customer_id not in self.U:
        self._init_customer(customer_id)
    if item_id not in self.V:
        self._init_item(item_id)

    # 🎯🎯🎯 Complete this --- compute z = dot product + biases 🎯🎯🎯
    z = ...
    return self._sigmoid(z)
 

### TODO 2: Implement Revenue-Optimized Recommendation

**Function**: `recommend(self, customer_id: int, k: int = 10) -> List[int]`

**Goal**: Recommend top-K items that maximize expected revenue for the customer.

**Mathematical Formula**:
$$\text{Score}(c,i) = (\text{price}_i - \text{cost}_i) \times \hat{\kappa}(c,i)$$

**Implementation Instructions**:
1. Get available items from `self.item_catalogue.get_available_items()`
2. For each item, compute expected revenue score using the formula above
3. Sort items by score in descending order
4. Return the top K item IDs as a list

**Business Logic**: We want to recommend items that customers are likely to buy (high κ) AND that generate high profit (high price - cost).

In [ ]:
def recommend(self, customer_id: int, k: int = 10) -> List[int]:
    """
    Recommend top-K items that maximize expected revenue for the customer.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Args:
        customer_id: ID of customer to recommend items for
        k: Number of items to recommend (default: 10)
        
    Returns:
        List[int]: List of top-K item IDs ranked by expected revenue
    """
    inventory = self.item_catalogue.get_available_items()
    if not inventory:
        return []
    
    scored_items = []
    # 🎯🎯🎯 Write your code here --- score and sort items 🎯🎯🎯
    for item in inventory:
        kappa = self._kappa_hat(customer_id, item.pid)
        # TODO: Your implementation of score here
        score = ...
        scored_items.append((item.pid, score))

    # Return top K
    scored_items.sort(key=lambda x: x[1], reverse=True)
    return [item_id for item_id, _ in scored_items[:min(k, len(scored_items))]]



### TODO 3: Implement Session Feedback Processing

**Function**: `update_from_session(self, customer_id: int, shown_items: List[int], purchased_items: List[int]) -> None`

**Goal**: Update model based on customer purchases and non-purchases during a session.

**Learning Strategy**:
- **Positive feedback**: Items the customer purchased → target = 1.0, weight = `w_purchase`
- **Negative feedback**: Items shown but not purchased → target = 0.0, weight = `w_neg`

**Implementation Instructions**:
1. For each item in `purchased_items`: call `_update_embeddings` with target=1.0 and weight=self.w_purchase
2. For each item in `shown_items` that is NOT in `purchased_items`: call `_update_embeddings` with target=0.0 and weight=self.w_neg

**Key Insight**: This implements implicit feedback learning - we learn from both what customers buy AND what they choose not to buy.

In [ ]:
def update_from_session(self, customer_id: int, shown_items: List[int], 
                        purchased_items: List[int]) -> None:
    """
    Update model based on customer purchases and non-purchases during a session.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Args:
        customer_id: ID of customer who had the session
        shown_items: List of item IDs shown to customer
        purchased_items: List of item IDs customer actually purchased
    """
    # 🎯🎯🎯 Write your code here --- call _update_embeddings for purchases and non-purchases 🎯🎯🎯
    target_purchased = ...
    target_not_purchased = ...

    for item_id in purchased_items:
        self._update_embeddings(customer_id, item_id, target=target_purchased
                                , weight=self.w_purchase)

    # Negative feedback: shown but not purchased items
    for item_id in shown_items:
        if item_id not in purchased_items:
            self._update_embeddings(customer_id, item_id, target=target_not_purchased
                                    , weight=self.w_neg)
 

### TODO 4: Implement Gradient Descent Updates  

**Function**: `_update_embeddings(self, customer_id: int, item_id: int, target: float, weight: float)`

**Goal**: Update customer and item embeddings using gradient descent on logistic loss.

**Mathematical Formulas**:

First, compute the prediction and error:
$$p = \hat{\kappa}(c,i)$$
$$\text{error} = \text{target} - p$$

Then update embeddings and biases using gradient descent:
$$U_c := U_c + \alpha w \cdot \mathrm{error} \cdot V_i - \alpha \lambda U_c$$
$$V_i := V_i + \alpha w \cdot \mathrm{error} \cdot U_c - \alpha \lambda V_i$$
$$b_c := b_c + \alpha w \cdot \mathrm{error} - \alpha \lambda b_c$$
$$b_i := b_i + \alpha w \cdot \mathrm{error} - \alpha \lambda b_i$$

where:
- $\alpha$ = learning rate (`self.lr`)
- $\lambda$ = regularization (`self.reg`)  
- $w$ = feedback weight (`weight` parameter)

**Implementation Instructions**:
1. Get current prediction using `_kappa_hat(customer_id, item_id)`
2. Compute prediction error: `target - prediction`
3. Update customer embedding `U[customer_id]` using the formula above
4. Update item embedding `V[item_id]` using the formula above
5. Update customer bias `b_c[customer_id]` using the formula above
6. Update item bias `b_i[item_id]` using the formula above

In [ ]:
def _update_embeddings(self, customer_id: int, item_id: int, target: float, weight: float):
    """
    Update customer and item embeddings using gradient descent on logistic loss.
    
    TODO: Implement this function following the instructions in the markdown cell above.
    
    Args:
        customer_id: ID of customer
        item_id: ID of item
        target: Target value (1.0 for purchase, 0.0 for non-purchase)
        weight: Learning weight (w_purchase or w_neg)
    """
    p = self._kappa_hat(customer_id, item_id)
    err = target - p

    # 🎯🎯🎯 Write your code here --- apply gradient descent updates to U, V, b_c, b_i 🎯🎯🎯
    # TODO: Your implementation of the gradient updates here
    self.U[customer_id] += ...
    self.V[item_id] += ...
    self.b_c[customer_id] += ...
    self.b_i[item_id] += ...


In [ ]:
#@title 🔗 Apply Methods to Agent — run, don't modify
# ▶️ Run this to apply your implementations to the RecommenderAgent class
# Apply the implemented methods to RecommenderAgent class
# (This connects your implementations to the actual agent)
RecommenderAgent._kappa_hat = _kappa_hat
RecommenderAgent.recommend = recommend
RecommenderAgent._update_embeddings = _update_embeddings
RecommenderAgent.update_from_session = update_from_session

print("✅ RecommenderAgent methods implemented and applied!")
print("Your CF agent is now ready for training and evaluation.")

### TODO Validation: Test Your Implementations

**Purpose**: Verify that your TODO implementations work correctly before running the full simulation.

**What this section does**:
- Creates a simple test agent to validate your implementations
- Checks that all functions run without errors
- Verifies basic functionality and output ranges
- Provides immediate feedback on implementation correctness

**Expected validation results**:
- All functions should execute without throwing exceptions
- κ predictions should be in [0,1] range  
- Recommendations should return item IDs
- Update functions should modify internal state

**Note**: This is a quick functionality test, not a performance evaluation. Full performance analysis comes after simulation.

In [ ]:
# ▶️ Run this to validate your implementations before training
validate_todo_implementations(customer_registry, item_catalogue, transaction_registry)

## 1. First Stage: Hyperparameter Configuration and Agent Setup

**What this stage accomplishes**: Configure your collaborative filtering model's hyperparameters and initialize the recommender agent for training. Your hyperparameter choices will directly impact model performance.

### **CF Hyperparameter Overview**

**Critical insight**: The performance of your CF model depends heavily on these hyperparameter choices. Each parameter controls different aspects of how your model learns customer preferences.

#### **Key Parameters to Configure**

**🔧 Embedding Dimension (`d`)**: Size of customer and item feature vectors
- **Range to explore**: 2-20 
- **Trade-off**: Complexity vs. overfitting risk

**🔧 Learning Rate (`lr`)**: Step size for gradient descent updates  
- **Range to explore**: 0.001-0.1
- **Trade-off**: Convergence speed vs. training stability

**🔧 Regularization (`reg`)**: Penalty for large embedding weights
- **Range to explore**: 1e-6 to 1e-2
- **Trade-off**: Overfitting prevention vs. underfitting risk

**🔧 Positive Feedback Weight (`w_purchase`)**: Learning strength from purchases
- **Range to explore**: 0.1-2.0
- **Impact**: How quickly model learns from successful recommendations

**🔧 Negative Feedback Weight (`w_neg`)**: Learning strength from non-purchases  
- **Range to explore**: 0.1-2.0
- **Impact**: How much model learns from rejected items

### **Experimentation Strategy**
- **Start with defaults** and observe baseline performance
- **Change one parameter at a time** to understand individual effects  
- **Use training curves and validation metrics** to guide your choices
- **Document what works** for different performance goals

### **Performance Indicators**
- **Training curves**: Should show smooth, steady improvement
- **κ Correlation**: Higher values indicate better prediction accuracy
- **Business metrics**: Revenue performance in final simulation

**🎯 Your mission**: Find hyperparameter combinations that maximize both prediction accuracy and business performance!

In [ ]:
# 🎯🎯🎯 Set your hyperparameters below --- experiment with different values! 🎯🎯🎯

# Embedding dimension (try values from 2-20)
d = None  # TODO: Choose your embedding dimension

# Learning rate (try values from 0.001-0.1) 
lr = None  # TODO: Choose your learning rate

# Regularization strength (try values from 1e-6 to 1e-2)
reg = None  # TODO: Choose your regularization parameter

# Positive feedback weight (try values from 0.1-2.0)
w_purchase = None  # TODO: Choose your positive feedback weight

# Negative feedback weight (try values from 0.1-2.0)  
w_neg = None  # TODO: Choose your negative feedback weight

In [ ]:
#@title 🤖 Agent Initialization — run, don't modify
# ▶️ Run this to create your recommender agent with the hyperparameters above


hyperparams_set = True
hyperparams_to_check = [
    (d, "d (embedding dimension)"),
    (lr, "lr (learning rate)"),
    (reg, "reg (regularization)"),
    (w_purchase, "w_purchase (positive feedback weight)"),
    (w_neg, "w_neg (negative feedback weight)")
]

print("🔍 Checking hyperparameters...")
for param_value, param_name in hyperparams_to_check:
    if param_value is None:
        print(f"⚠️  {param_name} needs to be specified (currently None)")
        hyperparams_set = False

if not hyperparams_set:
    print("❌ Please set all hyperparameter values in the cell above before proceeding.")
    print("Replace None with your chosen values based on the guidance provided.")
else:
    cf_agent = RecommenderAgent(customer_registry, item_catalogue, transaction_registry,
                                d=d, lr=lr, reg=reg, w_purchase=w_purchase, w_neg=w_neg)

    print("✅ CF Recommender Agent created successfully!")
    print()
    print("Current CF Hyperparameters:")
    print(f"Embedding dimension (d): {cf_agent.d}")
    print(f"Learning rate (lr): {cf_agent.lr}")
    print(f"Regularization (reg): {cf_agent.reg}")
    print(f"Positive feedback weight (w_purchase): {cf_agent.w_purchase}")
    print(f"Negative feedback weight (w_neg): {cf_agent.w_neg}")
    print()
    print("🚀 Ready to proceed with training!")

In [ ]:
#@title 📊 Ground Truth Computation — run, don't modify
# ▶️ Run this to compute the ground truth κ values used for validation
(customers, items, customer_ids, item_ids, customer_preferences,
 item_features, item_prices, item_revenues, true_kappa_matrix) = prepare_ground_truth_data(customer_registry, item_catalogue)

## 3. Second Stage: Run Training Simulation + Training Performance Analysis 🚀📊

**What this section shows**: Monitor how well your CF model learns during training by tracking two key metrics over 300 iterations:

1. **κ Prediction Error (MSE)** 📉: How accurately your model predicts purchase probabilities
2. **Training Revenue** 💰: Total profit generated during training simulations

**What to expect** 🎯:
- MSE should decrease from ~0.10–0.15 to ~0.03–0.04 over training ⬇️
- Revenue should stabilize around $26,000–$30,000 per iteration 📈
- Both metrics should show clear learning progress (not random fluctuation) ✨

**Interpreting the plots** 🔍:
- **Smooth decreasing MSE** = Model is successfully learning customer preferences 🧠
- **Stabilizing revenue** = Model converges to a consistent recommendation strategy 🎪
- **Final MSE ~0.02–0.04** = Reasonable prediction accuracy for this realistic training scenario ✅

**What the training simulation does** ⚙️:
The `train_cf_agent_with_simulation()` function implements realistic collaborative filtering training by:
- **Creating sparse training data** 📝: Each customer gets only 15 random items (not all 50) to simulate real-world constraints ❄️
- **Simulating purchase decisions** 🛒: Uses true κ probabilities to determine which recommended items customers "buy" 🎲
- **Updating embeddings** 🔄: CF agent learns from purchase/non-purchase feedback using gradient descent 📚
- **Testing generalization** 🧪: Evaluates κ prediction accuracy on ALL items (including unseen ones) every 10 iterations 🔭
- **Tracking learning progress** 📊: Records both prediction quality (MSE) and business performance (revenue) over time 📈

This approach tests whether your CF model truly learns generalizable customer preferences rather than just memorizing training data. 🧠💡

### **Note**: You do not need to modify anything in this cell, we simply show what the training procedure looks like

In [ ]:
# ▶️ Run this to define the training loop (no modifications needed)
def train_cf_agent_with_simulation(agent, n_iterations, customer_registry, item_catalogue,
                                   customer_ids, item_ids, true_kappa_matrix, items_per_customer):
    """
    Train CF agent by simulating customer sessions with realistic limited item exposure.

    Args:
        agent: The CF agent to train
        n_iterations: Number of training iterations
        customer_registry: Customer registry instance
        item_catalogue: Item catalogue instance
        customer_ids: List of customer IDs
        item_ids: List of item IDs
        true_kappa_matrix: True purchase probability matrix
        items_per_customer: Number of items each customer trains on

    Returns:
        tuple: (kappa_errors, revenues, customer_training_items)
    """
    customers = customer_registry.get_all_customers()

    # Creates customer-specific training item subsets (REALISTIC TRAINING)
    print(f"Creating customer-specific training subsets: {items_per_customer} items per customer")
    customer_training_items = {}
    np.random.seed(42)  # For reproducible training subsets

    for customer in customers:
        # Each customer gets their own random subset of items for training
        customer_training_items[customer.cid] = np.random.choice(
            item_ids,
            size=min(items_per_customer, len(item_ids)),
            replace=False
        ).tolist()

    # Prints training coverage statistics
    all_training_items = set()
    for items in customer_training_items.values():
        all_training_items.update(items)

    coverage_per_item = {}
    for item_id in item_ids:
        coverage_per_item[item_id] = sum(1 for items in customer_training_items.values() if item_id in items)

    avg_coverage = np.mean(list(coverage_per_item.values()))
    print(f"Training data sparsity: {len(all_training_items)}/{len(item_ids)} items seen")
    print(f"Average item coverage: {avg_coverage:.1f} customers per item")

    kappa_errors = []
    revenues = []

    for iteration in range(n_iterations):
        total_revenue = 0

        for customer in customers:
            customer_id = customer.cid

            recommendations = customer_training_items[customer_id]

            # Simulates purchases based on true κ
            purchased_items = []
            for item_id in recommendations:
                item_idx = item_ids.index(item_id) if item_id in item_ids else None
                customer_idx = customer_ids.index(customer_id)

                if item_idx is not None:
                    true_kappa = true_kappa_matrix[customer_idx, item_idx]
                    # Purchase with probability = true_kappa
                    if np.random.random() < true_kappa:
                        purchased_items.append(item_id)
                        # Adds to revenue
                        item = item_catalogue.get_item(item_id)
                        if item:
                            total_revenue += item.price - item.cost

            # Updates CF agent with feedback from limited training set
            agent.update_from_session(customer_id, recommendations, purchased_items)

        # Calculates κ prediction error (test on ALL items, not just training items)
        if iteration % 10 == 0:
            predicted_kappa = []
            true_kappa_flat = []

            for i, customer_id in enumerate(customer_ids[:10]):  # Sample for speed
                for j, item_id in enumerate(item_ids[:10]):
                    pred_kappa = agent._kappa_hat(customer_id, item_id)
                    true_kappa = true_kappa_matrix[i, j]
                    predicted_kappa.append(pred_kappa)
                    true_kappa_flat.append(true_kappa)

            mse = mean_squared_error(true_kappa_flat, predicted_kappa)
            kappa_errors.append(mse)
            revenues.append(total_revenue)

            if iteration % 20 == 0:
                print(f"Iteration {iteration}: MSE={mse:.4f}, Revenue=${total_revenue:.0f}")

    return kappa_errors, revenues, customer_training_items
# Trains CF agent with realistic limited item exposure
print("Training CF agent with realistic customer-specific item subsets...")
kappa_errors, revenues, customer_training_items = train_cf_agent_with_simulation(
    cf_agent, N_ITERATIONS, customer_registry, item_catalogue, 
    customer_ids, item_ids, true_kappa_matrix, ITEMS_PER_CUSTOMER)
print("Training simulation completed! 🎉📈")

## 4. Model Training Performance Analysis 📊

**What this section evaluates**: Test how well your trained CF model predicts purchase probabilities compared to ground truth. 🎯

**Key metrics to understand** 📈:

1. **Correlation (r)**: Measures how well predicted κ values match true κ values
   - **Target**: r > 0.7 indicates strong predictive ability ✅
   - **Realistic expectation**: > 0.7 with limited training data 📚

2. **Mean Squared Error (MSE)**: Average squared difference between predictions and truth
   - **Lower is better**: MSE ~0.02–0.04 is reasonable for this problem 🎪
   - **Context**: Perfect prediction would have MSE = 0 🎯

3. **Prediction range**: Check if model outputs reasonable probability values
   - **Should be**: Close to [0, 1] range like true κ values ✨
   - **Red flag**: Very narrow range indicates model isn't learning properly ⚠️

**Visualization insights** 🔍:
- **Scatter plot**: Points near diagonal line = good predictions 📍
- **Residual histogram**: Centered at 0 = unbiased predictions 🔔
- **Error vs truth**: Flat pattern = consistent accuracy across all κ values 📐

In [ ]:
# ▶️ Run this to plot the learning curves and track MSE over training
plot_learning_curves(kappa_errors, revenues, N_ITERATIONS)

## 5. Model Performance Validation 📊✨

**What this section shows**: Three diagnostic plots to evaluate how well
your trained CF model predicts purchase probabilities. 🎯

### **Plot explanations** 📈

**Plot 1 - Prediction Scatter Plot** 📍:
- **X-axis**: True κ(c,i) values from ground truth
- **Y-axis**: Predicted κ̂(c,i) values from your CF model
- **Red diagonal line**: Perfect prediction (where predicted = actual) ✅
- **What to look for**: Points clustered near the diagonal indicate good predictions 🎪

**Plot 2 - Error Distribution** 📊:
- **Shows**: Histogram of prediction errors (κ̂ - κ)
- **Red vertical line**: Zero error (perfect prediction) 🎯
- **What to look for**: Bell curve centered at 0 = unbiased predictions 🔔

**Plot 3 - Error vs Truth** 📉:
- **X-axis**: True κ(c,i) values
- **Y-axis**: Prediction errors (κ̂ - κ)
- **Red horizontal line**: Zero error
- **What to look for**: Random scatter around 0 = consistent accuracy across all κ values 🌟

**Success indicators** ✨:
- Strong correlation (r > 0.7) between true and predicted values 💪
- Low MSE (< 0.03) indicating accurate predictions 🎖️
- Unbiased errors centered around zero ⚖️
- Consistent performance across the full range of κ values 📐


In [ ]:
# ▶️ Run this to visualize how well your model predicts purchase probabilities
correlation, mse = analyze_and_plot_kappa_prediction_quality(cf_agent, customer_ids, item_ids, true_kappa_matrix)

## 6. Training Data Coverage Analysis 📊

**What this section reveals**: Visualize the realistic training constraints to understand why this evaluation is more challenging than typical academic scenarios. 🎯

**Key visualizations explained**: 🔍

1. **Training Matrix Heatmap**: Shows which customer-item pairs were used for training 🗺️
   - **Dark areas** = Training interactions ⚫
   - **Light areas** = Unseen during training (cold-start predictions) ⚪

2. **Item Coverage Distribution**: How many customers trained on each item 📈
   - **Expected**: Around 12 customers per item (from 15 items × 40 customers ÷ 50 items) 📊
   - **Reality**: Some items seen by more/fewer customers due to randomness 🎲

3. **Customer Training Set Size**: Should be uniform at 15 items per customer 👥
   - **Validates**: Each customer got exactly their allocated training subset ✅

4. **Training Efficiency Metrics**: 🚀
   - **600 training pairs** out of 2,000 possible (40×50) 💯
   - **All 50 items** appear in at least some customer's training set 🎁
   - **Even distribution** ensures no items are completely ignored 🔄

**Why this matters**: Your CF model must generalize from these 600 sparse interactions to predict all 2,000 possible customer-item combinations. This tests true learning vs memorization. 🧠💡

In [ ]:
# ▶️ Run this to analyze how many customers and items your model has learned from
plot_training_data_coverage(customer_ids, item_ids, customer_training_items, ITEMS_PER_CUSTOMER)

## 7. Baseline Comparison Strategy 🎯📊

**What this section does**: Compare your CF recommender against simple baseline strategies to validate that machine learning actually improves business outcomes. 🤖💡

**Baseline strategies explained**: 🎪

1. **Random Recommender**: 🎲
   - **Strategy**: Randomly selects K items for each customer
   - **Purpose**: Worst-case scenario baseline
   - **Expected behavior**: High conversion (customers buy whatever they see) but low revenue (no optimization) 📉

2. **Most Expensive Recommender**: 💎
   - **Strategy**: Always recommends the K most expensive items
   - **Purpose**: Simple heuristic that maximizes potential revenue per sale
   - **Expected behavior**: Lower conversion (expensive items harder to sell) but higher revenue when sales occur 💰

3. **CF (Trained)**: 🧠
   - **Strategy**: Your machine learning model that learns customer preferences
   - **Purpose**: Intelligent personalization based on collaborative filtering
   - **Expected behavior**: Best balance of conversion and revenue through personalized recommendations ⚖️

### Simulation 🚀
**Simulation approach**: Each strategy runs for 100 days, making daily recommendations to all customers and tracking business metrics. Your CF model continues learning from customer feedback during simulation. 📈

**Key plots explained**: 📊

**Revenue Analysis (Plots 1-4)**: 💵
- **Daily Revenue**: Shows consistency and trends over time 📅
- **Cumulative Revenue**: Total business impact over 100-day period 📈
- **Revenue Distribution**: Variability and risk assessment via box plots 📦
- **Total Revenue Comparison**: Final ranking of strategies 🏆

**Customer Experience (Plots 5, 9)**: 👥
- **Conversion Rate**: Percentage of recommendations that result in purchases 🎯
- **Overall Efficiency**: Success rate of the recommendation system ⚡
- **Trade-off insight**: Higher conversion doesn't always mean higher revenue 🔄

**Learning & Prediction Quality (Plots 6, 12)**: 📚
- **CF κ Prediction Error**: How prediction accuracy changes over time during simulation 🎓
- **Learning Progress**: Comparison between training performance and real-world application 🔬

**Business Insights (Plots 7-11)**: 💼
- **Revenue Growth Rate**: Sustainability and improvement trends 📈
- **Revenue per Customer**: Customer lifetime value implications 👤
- **Performance Stability**: Risk assessment through coefficient of variation ⚖️
- **Revenue per Customer**: Business scalability metrics 🔧

**What success looks like**: ✨
- **CF > Most Expensive > Random** in total revenue 🥇
- **Stable or improving** revenue trends over time 📊
- **Learning curve** showing continued CF improvement 🚀

In [ ]:
# ▶️ Run this to compare your agent against the random and most-expensive baselines 
print("Running detailed agent comparison simulation...")
print("This will take 2-3 minutes to complete all simulations.")
print()

# Initializes fresh agents for fair comparison
cf_agent # Uses trained CF agnet from before
random_agent = RandomRecommenderAgent(customer_registry, item_catalogue, transaction_registry)
expensive_agent = MostExpensiveRecommenderAgent(customer_registry, item_catalogue, transaction_registry)

# Runs detailed simulations for comparison
simulation_results = {}

print("\nRunning detailed simulations...")
simulation_results['CF (Trained)'] = detailed_agent_simulation(
    cf_agent, 'CF (Trained)', N_SIMULATION_DAYS, K, 
    customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)
simulation_results['Random'] = detailed_agent_simulation(
    random_agent, 'Random', N_SIMULATION_DAYS, K,
    customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)
simulation_results['Most Expensive'] = detailed_agent_simulation(
    expensive_agent, 'Most Expensive', N_SIMULATION_DAYS, K,
    customer_registry, item_catalogue, customer_ids, item_ids, true_kappa_matrix)

print("\n✅ Simulation complete! Results stored in 'simulation_results' variable.")

# Colors for consistency across plots
colors = {'CF (Trained)': '#2E86AB', 'Random': '#A23B72', 'Most Expensive': '#F18F01'}

# Create comprehensive comparison plots
plot_comprehensive_performance_analysis(simulation_results, colors, kappa_errors, N_ITERATIONS, customer_registry)

## 8. Executive Summary and Strategic Recommendations 📋✨

**What this section delivers**: Final business conclusions and actionable recommendations based on the comprehensive analysis above. 💼🎯

**Expected Key Findings**: 🔍

1. **CF Performance Validation** ✅:
   - CF should significantly outperform both Random and Most Expensive baselines 🏆
   - High κ correlation validates effective collaborative learning 🧠
   - Continued improvement during simulation proves adaptability 📈

2. **Business Trade-offs Revealed** ⚖️:
   - **CF**: Best revenue through personalized optimization 💰
   - **Most Expensive**: Reliable but sub-optimal heuristic approach 💎
   - **Random**: High conversion but poor revenue optimization 🎲

**Strategic Implications** 💡: These results validate collaborative filtering as a superior approach for revenue optimization while maintaining reasonable customer experience, justifying machine learning investment over simple heuristic rules. 🚀💼

In [ ]:
# ▶️ Run this to generate the executive summary and business recommendations
if 'correlation' not in dir() or 'mse' not in dir():
    correlation, mse = analyze_and_plot_kappa_prediction_quality(cf_agent, customer_ids, item_ids, true_kappa_matrix)
print_comprehensive_analysis(simulation_results, correlation, mse, customer_registry)

# Next Steps: Optimize Your Hyperparameters 🚀✨

Now that you've trained your model, it's time to experiment and improve! Here are some suggestions:

## If You Haven't Passed the Baselines Yet ❌
- 🔧 Go back and adjust your **hyperparameters** (learning rate, embedding dimension, regularization strength...)

## If You've Already Passed the Baselines ✅
- 🔍 **Fine-tune** your hyperparameters for even better performance
- ⏹️ Implement **early stopping** or **learning rate scheduling**
- 🧠 Analyze **error patterns** to identify areas for improvement
- ⚙️ Test different **optimization algorithms** (Adam, SGD, RMSprop, etc.)
- 🤝 Try **ensemble methods** combining multiple models (**optional, advanced**): You can train multiple CF models with different hyperparameters and average their predictions. This is by no means needed for the assignment but can improve robustness.


## Tips for Experimentation 📝
- 🔁 Change **one hyperparameter at a time** to isolate its effect
- 🗒️ Keep **notes** on what you tried and the results
- 📊 Use **validation metrics** to track progress
- 🔁 Don't forget to **re-run your baseline** to confirm reproducibility

Good luck improving your model! 🚀💡